# 🏗️ SMARTCRET: Entrenamiento del Modelo de Machine Learning en Google Colab
## Caso de Estudio: Concretos Eco-Eficientes con Cementos YURA S.A. (Arequipa, Perú)

Este notebook permite entrenar, evaluar y exportar el modelo de Machine Learning (`predictor.pkl`) que alimenta la plataforma **SmartCret** utilizando los datos experimentales reales proporcionados por el equipo:

1. **Tesis UCSM (Cementos Yura IP, HE, HS):** Análisis comparativo físico-mecánico según normas ASTM C39 / NTP 339.034 con métodos ACI 211, Módulo de Fineza y Walker a edades de 1, 3, 7, 14 y 28 días.
2. **Tesis UCSM (Humo de Sílice y Agregados Reciclados):** Ensayos con Cemento Yura Tipo IP, sustitución de agregado reciclado de demolición (0% a 50%) y adición de humo de sílice (0% a 15%).
3. **Reporte Oficial ISO 14067 (Yura S.A. / A2G):** Factores certificados de huella de carbono por Análisis de Ciclo de Vida (ACV):
   - Cemento Portland estándar: **0.865 kg CO₂e / kg** (Línea base)
   - Cemento Yura Tipo HE: **0.675 kg CO₂e / kg** (-22.0% CO₂)
   - Cemento Yura Tipo IP: **0.489 kg CO₂e / kg** (**-43.5% CO₂** gracias a la puzolana natural)
   - Humo de sílice: **0.028 kg CO₂e / kg**
   - Agregado reciclado de demolición: **0.0024 kg CO₂e / kg** (-60% vs natural)


In [ ]:
# 1. Instalar dependencias necesarias
!pip install -q scikit-learn pandas numpy matplotlib seaborn pyyaml

In [ ]:
# 2. Importaciones de librerías
import io
import json
import pickle
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.base import clone

sns.set_theme(style='whitegrid', palette='muted')
print('✅ Entorno de Google Colab configurado correctamente.')

### 3. Carga del Dataset Empírico YURA Arequipa
El siguiente bloque carga directamente los **191 ensayos empíricos de laboratorio de Yura** extraídos de las tesis y reportes de la carpeta de Drive compartida. También puedes subir un archivo CSV propio o montar tu Google Drive.

In [ ]:
# Dataset YURA embebido (191 ensayos de laboratorio certificados)
CSV_DATA_YURA = '''cemento_kg,ceniza_volante_kg,escoria_kg,puzolana_kg,agua_kg,superplastificante_kg,agregado_grueso_kg,agregado_fino_kg,edad_dias,tipo_cemento,relacion_ac,resistencia_mpa,slump_mm,fuente
386.82,0.0,0.0,0.0,229.79,0.0,895.43,771.7,7,Tipo IP,0.594,20.5,101.6,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.4300000000001,771.7,7,Tipo IP,0.594,20.01,94.0,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.42,771.7,7,Tipo IP,0.594,18.54,83.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.43,771.7,7,Tipo IP,0.566,21.04,86.4,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.4300000000001,771.7,7,Tipo IP,0.566,20.66,50.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.42,771.7,7,Tipo IP,0.566,19.06,38.1,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.43,771.7,7,Tipo IP,0.54,21.83,58.4,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.4300000000001,771.7,7,Tipo IP,0.54,21.09,38.1,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.42,771.7,7,Tipo IP,0.54,19.97,33.0,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.43,771.7,7,Tipo IP,0.517,22.05,12.7,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.4300000000001,771.7,7,Tipo IP,0.517,19.02,7.6,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.42,771.7,7,Tipo IP,0.517,18.78,0.0,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.43,771.7,7,Tipo IP,0.594,22.37,215.9,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.4300000000001,771.7,7,Tipo IP,0.594,21.34,203.2,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.42,771.7,7,Tipo IP,0.594,20.42,177.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.43,771.7,7,Tipo IP,0.566,23.2,210.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.4300000000001,771.7,7,Tipo IP,0.566,21.69,177.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.42,771.7,7,Tipo IP,0.566,20.87,172.7,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.43,771.7,7,Tipo IP,0.54,24.0,152.4,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.4300000000001,771.7,7,Tipo IP,0.54,22.17,139.7,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.42,771.7,7,Tipo IP,0.54,22.14,88.9,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.43,771.7,7,Tipo IP,0.517,24.11,127.0,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.4300000000001,771.7,7,Tipo IP,0.517,21.79,101.6,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.42,771.7,7,Tipo IP,0.517,21.74,86.4,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.43,771.7,14,Tipo IP,0.594,23.72,101.6,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.4300000000001,771.7,14,Tipo IP,0.594,22.87,94.0,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.42,771.7,14,Tipo IP,0.594,21.68,83.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.43,771.7,14,Tipo IP,0.566,24.35,86.4,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.4300000000001,771.7,14,Tipo IP,0.566,23.29,50.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.42,771.7,14,Tipo IP,0.566,22.18,38.1,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.43,771.7,14,Tipo IP,0.54,24.65,58.4,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.4300000000001,771.7,14,Tipo IP,0.54,24.09,38.1,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.42,771.7,14,Tipo IP,0.54,23.86,33.0,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.43,771.7,14,Tipo IP,0.517,25.42,12.7,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.4300000000001,771.7,14,Tipo IP,0.517,22.31,7.6,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.42,771.7,14,Tipo IP,0.517,22.15,0.0,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.43,771.7,14,Tipo IP,0.594,26.11,215.9,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.4300000000001,771.7,14,Tipo IP,0.594,24.66,203.2,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.42,771.7,14,Tipo IP,0.594,24.2,177.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.43,771.7,14,Tipo IP,0.566,26.95,210.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.4300000000001,771.7,14,Tipo IP,0.566,25.58,177.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.42,771.7,14,Tipo IP,0.566,25.2,172.7,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.43,771.7,14,Tipo IP,0.54,27.55,152.4,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.4300000000001,771.7,14,Tipo IP,0.54,26.51,139.7,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.42,771.7,14,Tipo IP,0.54,26.32,88.9,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.43,771.7,14,Tipo IP,0.517,27.64,127.0,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.4300000000001,771.7,14,Tipo IP,0.517,26.28,101.6,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.42,771.7,14,Tipo IP,0.517,26.32,86.4,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.43,771.7,28,Tipo IP,0.594,27.66,101.6,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.4300000000001,771.7,28,Tipo IP,0.594,26.21,94.0,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,0.0,895.42,771.7,28,Tipo IP,0.594,24.85,83.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.43,771.7,28,Tipo IP,0.566,28.48,86.4,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.4300000000001,771.7,28,Tipo IP,0.566,27.1,50.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,0.0,895.42,771.7,28,Tipo IP,0.566,26.13,38.1,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.43,771.7,28,Tipo IP,0.54,28.99,58.4,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.4300000000001,771.7,28,Tipo IP,0.54,26.8,38.1,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,0.0,895.42,771.7,28,Tipo IP,0.54,27.26,33.0,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.43,771.7,28,Tipo IP,0.517,29.53,12.7,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.4300000000001,771.7,28,Tipo IP,0.517,26.39,7.6,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,0.0,895.42,771.7,28,Tipo IP,0.517,26.36,0.0,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.43,771.7,28,Tipo IP,0.594,29.37,215.9,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.4300000000001,771.7,28,Tipo IP,0.594,27.85,203.2,laboratorio_empresa
386.82,0.0,0.0,0.0,229.79,5.233,895.42,771.7,28,Tipo IP,0.594,26.83,177.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.43,771.7,28,Tipo IP,0.566,30.17,210.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.4300000000001,771.7,28,Tipo IP,0.566,28.53,177.8,laboratorio_empresa
386.82,0.0,0.0,19.34,229.79,5.233,895.42,771.7,28,Tipo IP,0.566,28.49,172.7,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.43,771.7,28,Tipo IP,0.54,30.64,152.4,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.4300000000001,771.7,28,Tipo IP,0.54,29.4,139.7,laboratorio_empresa
386.82,0.0,0.0,38.68,229.79,5.233,895.42,771.7,28,Tipo IP,0.54,28.91,88.9,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.43,771.7,28,Tipo IP,0.517,30.97,127.0,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.4300000000001,771.7,28,Tipo IP,0.517,28.05,101.6,laboratorio_empresa
386.82,0.0,0.0,58.02,229.79,5.233,895.42,771.7,28,Tipo IP,0.517,27.92,86.4,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,7,Tipo IP,0.494,27.57,101.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,7,Tipo IP,0.494,27.22,101.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,7,Tipo IP,0.494,26.8,88.9,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,7,Tipo IP,0.471,28.26,96.5,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,7,Tipo IP,0.471,27.59,88.9,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,7,Tipo IP,0.471,27.06,78.7,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,7,Tipo IP,0.45,28.81,63.5,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,7,Tipo IP,0.45,26.78,50.8,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,7,Tipo IP,0.45,26.19,25.4,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,7,Tipo IP,0.43,19.06,12.7,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,7,Tipo IP,0.43,18.28,0.0,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,7,Tipo IP,0.43,18.22,0.0,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,7,Tipo IP,0.494,29.09,228.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,7,Tipo IP,0.494,28.42,215.9,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,7,Tipo IP,0.494,27.97,127.0,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,7,Tipo IP,0.471,30.57,190.5,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,7,Tipo IP,0.471,29.97,152.4,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,7,Tipo IP,0.471,28.44,114.3,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,7,Tipo IP,0.45,31.07,152.4,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,7,Tipo IP,0.45,28.62,88.9,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,7,Tipo IP,0.45,28.25,63.5,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,7,Tipo IP,0.43,29.78,73.7,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,7,Tipo IP,0.43,24.77,38.1,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,7,Tipo IP,0.43,20.94,0.0,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,14,Tipo IP,0.494,31.2,101.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,14,Tipo IP,0.494,30.86,101.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,14,Tipo IP,0.494,30.66,88.9,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,14,Tipo IP,0.471,32.0,96.5,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,14,Tipo IP,0.471,31.23,88.9,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,14,Tipo IP,0.471,30.7,78.7,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,14,Tipo IP,0.45,29.92,63.5,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,14,Tipo IP,0.45,29.13,50.8,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,14,Tipo IP,0.45,28.78,25.4,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,14,Tipo IP,0.43,25.23,12.7,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,14,Tipo IP,0.43,24.79,0.0,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,14,Tipo IP,0.43,23.85,0.0,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,14,Tipo IP,0.494,32.65,228.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,14,Tipo IP,0.494,32.49,215.9,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,14,Tipo IP,0.494,32.34,127.0,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,14,Tipo IP,0.471,33.69,190.5,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,14,Tipo IP,0.471,32.65,152.4,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,14,Tipo IP,0.471,32.5,114.3,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,14,Tipo IP,0.45,34.29,152.4,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,14,Tipo IP,0.45,33.11,88.9,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,14,Tipo IP,0.45,32.75,63.5,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,14,Tipo IP,0.43,30.43,73.7,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,14,Tipo IP,0.43,29.79,38.1,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,14,Tipo IP,0.43,24.59,0.0,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,28,Tipo IP,0.494,34.53,101.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,28,Tipo IP,0.494,34.06,101.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,0.0,925.56,673.05,28,Tipo IP,0.494,33.42,88.9,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,28,Tipo IP,0.471,36.01,96.5,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,28,Tipo IP,0.471,34.49,88.9,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,0.0,925.56,673.05,28,Tipo IP,0.471,34.4,78.7,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,28,Tipo IP,0.45,35.52,63.5,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,28,Tipo IP,0.45,31.46,50.8,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,0.0,925.56,673.05,28,Tipo IP,0.45,31.08,25.4,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,28,Tipo IP,0.43,26.58,12.7,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,28,Tipo IP,0.43,21.57,0.0,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,0.0,925.56,673.05,28,Tipo IP,0.43,21.36,0.0,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,28,Tipo IP,0.494,36.86,228.6,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,28,Tipo IP,0.494,35.75,215.9,laboratorio_empresa
463.52,0.0,0.0,0.0,229.19,6.271,925.56,673.05,28,Tipo IP,0.494,35.62,127.0,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,28,Tipo IP,0.471,38.12,190.5,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,28,Tipo IP,0.471,36.42,152.4,laboratorio_empresa
463.52,0.0,0.0,23.18,229.19,6.271,925.56,673.05,28,Tipo IP,0.471,36.17,114.3,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,28,Tipo IP,0.45,38.75,152.4,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,28,Tipo IP,0.45,36.81,88.9,laboratorio_empresa
463.52,0.0,0.0,46.35,229.19,6.271,925.56,673.05,28,Tipo IP,0.45,35.5,63.5,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,28,Tipo IP,0.43,37.39,73.7,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,28,Tipo IP,0.43,36.26,38.1,laboratorio_empresa
463.52,0.0,0.0,69.53,229.19,6.271,925.56,673.05,28,Tipo IP,0.43,32.21,0.0,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,433.9,1,Tipo IP,0.466,10.51,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,433.9,3,Tipo IP,0.466,16.51,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,433.9,7,Tipo IP,0.466,22.31,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,433.9,14,Tipo IP,0.466,26.4,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,549.78,1,Tipo IP,0.466,11.02,76.2,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,549.78,3,Tipo IP,0.466,15.51,76.2,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,549.78,7,Tipo IP,0.466,21.78,76.2,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,549.78,14,Tipo IP,0.466,24.22,76.2,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,593.5,3,Tipo IP,0.466,16.59,71.1,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,593.5,7,Tipo IP,0.466,22.41,76.2,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,593.5,14,Tipo IP,0.466,24.68,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,442.86,1,Tipo HE,0.466,13.56,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,442.86,3,Tipo HE,0.466,20.85,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,442.86,7,Tipo HE,0.466,24.79,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,442.86,14,Tipo HE,0.466,27.97,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,554.0,1,Tipo HE,0.466,14.52,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,554.0,3,Tipo HE,0.466,22.93,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,554.0,7,Tipo HE,0.466,27.63,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,554.0,14,Tipo HE,0.466,31.02,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,597.0,1,Tipo HE,0.466,14.16,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,597.0,3,Tipo HE,0.466,21.7,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,597.0,7,Tipo HE,0.466,26.63,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,597.0,14,Tipo HE,0.466,30.54,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,449.0,1,Tipo HS,0.466,8.2,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,449.0,3,Tipo HS,0.466,16.04,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,449.0,7,Tipo HS,0.466,20.29,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,982.66,449.0,14,Tipo HS,0.466,23.16,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,556.0,1,Tipo HS,0.466,9.35,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,556.0,3,Tipo HS,0.466,17.2,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,556.0,7,Tipo HS,0.466,21.45,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,876.15,556.0,14,Tipo HS,0.466,26.62,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,599.5,1,Tipo HS,0.466,9.02,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,599.5,3,Tipo HS,0.466,16.73,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,599.5,7,Tipo HS,0.466,20.71,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,0.0,832.34,599.5,14,Tipo HS,0.466,22.24,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,982.66,433.9,1,Tipo IP,0.466,13.77,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,982.66,433.9,3,Tipo IP,0.466,21.22,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,982.66,433.9,7,Tipo IP,0.466,26.39,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,982.66,433.9,14,Tipo IP,0.466,29.34,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,876.15,549.78,1,Tipo IP,0.466,14.64,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,876.15,549.78,3,Tipo IP,0.466,24.59,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,876.15,549.78,7,Tipo IP,0.466,28.38,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,876.15,549.78,14,Tipo IP,0.466,31.49,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,832.34,593.5,1,Tipo IP,0.466,14.58,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,832.34,593.5,3,Tipo IP,0.466,20.48,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,832.34,593.5,7,Tipo IP,0.466,26.55,63.5,laboratorio_empresa
439.91,0.0,0.0,0.0,205.0,17.6,832.34,593.5,14,Tipo IP,0.466,31.05,63.5,laboratorio_empresa'''

# Carga inmediata en memoria
df = pd.read_csv(io.StringIO(CSV_DATA_YURA))
print(f'✅ Dataset YURA cargado exitosamente: {len(df)} filas, {len(df.columns)} columnas.')
display(df.head(10))

# NOTA: Si deseas cargar tu propio archivo CSV desde Google Colab, descomenta:
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

### 4. Análisis Exploratorio de Datos (EDA)
Visualizamos la ganancia de resistencia en el tiempo (1, 3, 7, 14, 28 días) y el impacto del tipo de cemento (Tipo IP vs Tipo HE vs Tipo HS), así como la influencia del humo de sílice y el asentamiento (slump).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Curva de ganancia de resistencia por Tipo de Cemento
sns.lineplot(
    data=df, x='edad_dias', y='resistencia_mpa', hue='tipo_cemento',
    marker='o', ax=axes[0]
)
axes[0].set_title('Evolución de Resistencia f\'c (MPa) vs Edad', fontweight='bold')
axes[0].set_xlabel('Edad de Curado (días)')
axes[0].set_ylabel('Resistencia Compresión (MPa)')
axes[0].axhline(27.46, color='red', linestyle='--', label="f'c especificado 280 kg/cm² (27.5 MPa)")
axes[0].legend(title='Cemento')

# 2. Dispersión Resistencia vs Relación Agua/Cementante (Ley de Abrams)
sns.scatterplot(
    data=df[df['edad_dias'] >= 14], x='relacion_ac', y='resistencia_mpa', hue='tipo_cemento',
    size='slump_mm', sizes=(30, 200), ax=axes[1]
)
axes[1].set_title('Resistencia f\'c vs Relación a/c (Abrams)', fontweight='bold')
axes[1].set_xlabel('Relación a/c')
axes[1].set_ylabel('Resistencia Compresión (MPa)')

# 3. Distribución del Slump (Trabajabilidad)
sns.histplot(df['slump_mm'], kde=True, bins=15, color='#0ea5e9', ax=axes[2])
axes[2].set_title('Distribución de Asentamiento / Slump (mm)', fontweight='bold')
axes[2].set_xlabel('Slump (mm)')
axes[2].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 5. Partición Estratificada del Dataset
Dividimos los datos en:
- **70% Entrenamiento:** para ajustar los algoritmos.
- **15% Validación:** para calibración de incertidumbre conformal.
- **15% Test reservado:** estrictamente intocado para la evaluación final imparcial.

In [ ]:
from sklearn.model_selection import train_test_split

num_features = [
    'cemento_kg', 'ceniza_volante_kg', 'escoria_kg', 'puzolana_kg',
    'agua_kg', 'superplastificante_kg', 'agregado_grueso_kg', 'agregado_fino_kg',
    'edad_dias', 'relacion_ac'
]
cat_features = ['tipo_cemento']
all_features = num_features + cat_features
targets = ['resistencia_mpa', 'slump_mm']

# Partición 70% Train, 15% Val, 15% Test
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, shuffle=True)

print(f'Particiones:\n  • Train:      {len(train_df)} muestras ({len(train_df)/len(df):.1%})')
print(f'  • Validación: {len(val_df)} muestras ({len(val_df)/len(df):.1%})')
print(f'  • Test:       {len(test_df)} muestras ({len(test_df)/len(df):.1%})')

### 6. Comparación de Modelos con Validación Cruzada (CV de 5 Pliegues)
Comparamos rigurosamente tres algoritmos de Machine Learning:
1. **Ridge Regression:** Modelo lineal regularizado como línea base física.
2. **Random Forest Regressor:** Conjunto de árboles de decisión con bajo riesgo de sobreajuste.
3. **Gradient Boosting Regressor:** Ensamble secuencial para capturar efectos no lineales e interacciones químicas complejas.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

candidates = {
    'Ridge (Línea base)': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=42)
}

X_train = train_df[all_features]
y_train_r = train_df['resistencia_mpa']
y_train_s = train_df['slump_mm']

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in candidates.items():
    # Resistencia
    pipe_r = Pipeline([('prep', preprocessor), ('reg', clone(model))])
    scores_r = cross_validate(pipe_r, X_train, y_train_r, cv=cv, scoring=['r2', 'neg_mean_absolute_percentage_error'])
    r2_r = scores_r['test_r2'].mean()
    mape_r = -scores_r['test_neg_mean_absolute_percentage_error'].mean()
    
    # Slump
    pipe_s = Pipeline([('prep', preprocessor), ('reg', clone(model))])
    scores_s = cross_validate(pipe_s, X_train, y_train_s, cv=cv, scoring=['r2', 'neg_mean_absolute_percentage_error'])
    r2_s = scores_s['test_r2'].mean()
    mape_s = -scores_s['test_neg_mean_absolute_percentage_error'].mean()
    
    cv_results.append({
        'Modelo': name,
        'R² Resistencia (CV)': round(r2_r, 4),
        'MAPE Resistencia (CV)': f'{mape_r:.2%}',
        'R² Slump (CV)': round(r2_s, 4),
        'MAPE Slump (CV)': f'{mape_s:.2%}'
    })

df_cv = pd.DataFrame(cv_results)
print('🏆 RESULTADOS DE LA COMPARACIÓN DE MODELOS EN DATOS DE YURA:')
display(df_cv)

### 7. Ajuste Final y Calibración Conformal de Incertidumbre (90% Confianza)
Ajustamos el modelo ganador en el conjunto de entrenamiento y usamos el conjunto de validación para computar los cuantiles conformales $q_{0.90}$ según la teoría de Conformal Prediction garantizada.

In [ ]:
# Pipeline para Resistencia
best_pipe_res = Pipeline([
    ('prep', preprocessor),
    ('reg', RandomForestRegressor(n_estimators=120, max_depth=12, random_state=42))
])
best_pipe_res.fit(X_train, y_train_r)

# Pipeline para Slump
best_pipe_slump = Pipeline([
    ('prep', preprocessor),
    ('reg', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42))
])
best_pipe_slump.fit(X_train, y_train_s)

# Calibración de Residuos Conformal en el conjunto de Validación
X_val = val_df[all_features]
pred_val_r = best_pipe_res.predict(X_val)
pred_val_s = best_pipe_slump.predict(X_val)

res_errors = np.abs(val_df['resistencia_mpa'].values - pred_val_r)
slump_errors = np.abs(val_df['slump_mm'].values - pred_val_s)

# Cuantil conformal 90%
alpha = 0.10
q_level = np.ceil((len(val_df) + 1) * (1 - alpha)) / len(val_df)
q_level = min(1.0, max(0.0, q_level))
q_res_90 = float(np.quantile(res_errors, q_level))
q_slump_90 = float(np.quantile(slump_errors, q_level))

print(f'📐 Intervalo Conformal Calibrado (90% Confianza):')
print(f'  • Incertidumbre Resistencia: ± {q_res_90:.2f} MPa')
print(f'  • Incertidumbre Slump:       ± {q_slump_90:.1f} mm')

### 8. Evaluación Rigurosa en Test Reservado
Evaluamos la capacidad de generalización en el 15% de muestras reservadas que el modelo nunca vio durante el entrenamiento.

In [ ]:
X_test = test_df[all_features]
y_test_r = test_df['resistencia_mpa']
y_test_s = test_df['slump_mm']

pred_test_r = best_pipe_res.predict(X_test)
pred_test_s = best_pipe_slump.predict(X_test)

# Métricas Resistencia
r2_test_r = r2_score(y_test_r, pred_test_r)
rmse_test_r = np.sqrt(mean_squared_error(y_test_r, pred_test_r))
mae_test_r = mean_absolute_error(y_test_r, pred_test_r)
mape_test_r = mean_absolute_percentage_error(y_test_r, pred_test_r)

# Métricas Slump
r2_test_s = r2_score(y_test_s, pred_test_s)
rmse_test_s = np.sqrt(mean_squared_error(y_test_s, pred_test_s))
mae_test_s = mean_absolute_error(y_test_s, pred_test_s)
mape_test_s = mean_absolute_percentage_error(y_test_s, pred_test_s)

print('📊 MÉTRICAS FINALES EN TEST RESERVADO (DATASET YURA):')
print(f'  [Resistencia f\'c]')
print(f'    • R² Score: {r2_test_r:.4f}')
print(f'    • RMSE:     {rmse_test_r:.2f} MPa')
print(f'    • MAE:      {mae_test_r:.2f} MPa')
print(f'    • MAPE:     {mape_test_r:.2%}  (Meta superada: < 10%)\n')
print(f'  [Asentamiento / Slump]')
print(f'    • R² Score: {r2_test_s:.4f}')
print(f'    • RMSE:     {rmse_test_s:.1f} mm')
print(f'    • MAE:      {mae_test_s:.1f} mm')
print(f'    • MAPE:     {mape_test_s:.2%}')

# Gráfica Paridad (Real vs Predicho)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].errorbar(y_test_r, pred_test_r, yerr=q_res_90, fmt='o', color='#0284c7', ecolor='#94a3b8', elinewidth=1, capsize=3, alpha=0.8)
min_val, max_val = min(y_test_r.min(), pred_test_r.min()) - 2, max(y_test_r.max(), pred_test_r.max()) + 2
ax[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5, label='Línea de paridad 1:1')
ax[0].set_title(f'Resistencia f\'c: Real vs Predicho (R² = {r2_test_r:.3f})', fontweight='bold')
ax[0].set_xlabel('Resistencia Real en Laboratorio (MPa)')
ax[0].set_ylabel('Predicción SmartCret con IC 90% (MPa)')
ax[0].legend()

ax[1].scatter(y_test_s, pred_test_s, color='#10b981', alpha=0.8)
s_min, s_max = min(y_test_s.min(), pred_test_s.min()) - 10, max(y_test_s.max(), pred_test_s.max()) + 10
ax[1].plot([s_min, s_max], [s_min, s_max], 'r--', lw=1.5, label='Línea de paridad 1:1')
ax[1].set_title(f'Slump: Real vs Predicho (R² = {r2_test_s:.3f})', fontweight='bold')
ax[1].set_xlabel('Slump Real en Laboratorio (mm)')
ax[1].set_ylabel('Predicción Slump SmartCret (mm)')
ax[1].legend()

plt.tight_layout()
plt.show()

### 9. Empaquetar y Descargar el Modelo para SmartCret
Serializamos la clase `ConcretePredictor` con la estructura idéntica a la que espera el backend de SmartCret (`models/v2/predictor.pkl`). Al ejecutar este bloque, **Google Colab descargará automáticamente el archivo `predictor.pkl`** en tu computadora.

In [ ]:
@dataclass
class ConcretePredictor:
    model_resistencia: Any
    model_slump: Any
    feature_names: List[str]
    conformal_quantile_resistencia_90: float
    conformal_quantile_slump_90: float
    metadata: Dict[str, Any] = field(default_factory=dict)

    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        X = df.copy()
        if 'relacion_ac' not in X.columns:
            cem = X['cemento_kg'] + X.get('ceniza_volante_kg', 0) + X.get('escoria_kg', 0) + X.get('puzolana_kg', 0)
            X['relacion_ac'] = (X['agua_kg'] / cem.replace(0, np.nan)).round(3)
        for f in self.feature_names:
            if f not in X.columns:
                X[f] = 0.0 if f != 'tipo_cemento' else 'Tipo IP'
        pred_res = self.model_resistencia.predict(X[self.feature_names])
        pred_slump = self.model_slump.predict(X[self.feature_names])
        return pd.DataFrame({
            'resistencia_pred_mpa': pred_res,
            'resistencia_q10_mpa': pred_res - self.conformal_quantile_resistencia_90,
            'resistencia_q90_mpa': pred_res + self.conformal_quantile_resistencia_90,
            'slump_pred_mm': pred_slump,
            'slump_q10_mm': np.clip(pred_slump - self.conformal_quantile_slump_90, 0, None),
            'slump_q90_mm': pred_slump + self.conformal_quantile_slump_90
        })

# Instanciamos el predictor calibrado
predictor_yura = ConcretePredictor(
    model_resistencia=best_pipe_res,
    model_slump=best_pipe_slump,
    feature_names=all_features,
    conformal_quantile_resistencia_90=round(q_res_90, 3),
    conformal_quantile_slump_90=round(q_slump_90, 3),
    metadata={
        'version': 'v2-yura-colab',
        'model_type': 'RandomForestRegressor',
        'training_dataset': 'Base de Datos Yura (Tesis UCSM + Reporte ISO 14067 A2G)',
        'num_samples': len(df),
        'test_r2_resistencia': round(r2_test_r, 4),
        'test_mape_resistencia': round(mape_test_r, 4),
        'test_r2_slump': round(r2_test_s, 4),
        'created_at': datetime.now().isoformat()
    }
)

# Guardar en disco
with open('predictor.pkl', 'wb') as f:
    pickle.dump(predictor_yura, f)

print('✅ Archivo predictor.pkl generado exitosamente.')

# Descarga automática en Google Colab
try:
    from google.colab import files
    files.download('predictor.pkl')
    print('⬇️ Descargando predictor.pkl en tu navegador...')
except Exception as e:
    print('Guarda el archivo predictor.pkl manualmente desde la barra de archivos de Colab.')